# Exemplo: Previsão multivariada
--------------------

Este exemplo mostra como usar o ExperionML com um conjunto de dados de séries temporais multivariadas com variáveis exógenas.

Importe o conjunto de dados macroeconômicos de [sktime.datasets](https://www.sktime.net/en/latest/api_reference/auto_generated/sktime.datasets.load_macroeconomic.html). Este é um pequeno conjunto de dados que mede dados macroeconômicos dos EUA entre 1959 e 2009.

## Carregar os dados

In [1]:
# Import packages
import numpy as np
from sktime.datasets import load_macroeconomic
from experionml import ExperionMLForecaster

In [2]:
# Carregar os dados
X = load_macroeconomic()

print(X)

          realgdp  realcons   realinv  realgovt  realdpi      cpi      m1  \
Period                                                                      
1959Q1   2710.349    1707.4   286.898   470.045   1886.9   28.980   139.7   
1959Q2   2778.801    1733.7   310.859   481.301   1919.7   29.150   141.7   
1959Q3   2775.488    1751.8   289.226   491.260   1916.4   29.350   140.5   
1959Q4   2785.204    1753.7   299.356   484.052   1931.3   29.370   140.0   
1960Q1   2847.699    1770.5   331.722   462.199   1955.5   29.540   139.6   
...           ...       ...       ...       ...      ...      ...     ...   
2008Q3  13324.600    9267.7  1990.693   991.551   9838.3  216.889  1474.7   
2008Q4  13141.920    9195.3  1857.661  1007.273   9920.4  212.174  1576.5   
2009Q1  12925.410    9209.2  1558.494   996.287   9926.4  212.671  1592.8   
2009Q2  12901.504    9189.0  1456.678  1023.528  10077.5  214.469  1653.6   
2009Q3  12990.341    9256.0  1486.398  1044.088  10040.6  216.385  1673.9   

## Analisar os dados

In [3]:
# Especificamos as duas últimas colunas como colunas alvo
experionml = ExperionMLForecaster(X, y=(-2, -1), verbose=2, random_state=1)

<< ================== ExperionML ================== >>

Configuração ==================== >>
Tarefa do algoritmo: Multivariate forecast.

Estatísticas do conjunto de dados ==================== >>
Formato: (203, 12)
Tamanho do conjunto de train: 163
 --> De: 1959Q1  Até: 1999Q3
Tamanho do conjunto de test: 40
 --> De: 1999Q4  Até: 2009Q3
-------------------------------------
Memória: 29.41 kB
Escalonado: False
Valores atípicos: 9 (0.5%)



In [4]:
experionml.dataset

,realgdp,realcons,realinv,realgovt,realdpi,cpi,m1,tbilrate,unemp,pop,infl,realint
Period,,,,,,,,,,,,
1959Q1,2710.349,1707.4,286.898,470.045,1886.9,28.980,139.7,2.82,5.8,177.146,0.00,0.00
1959Q2,2778.801,1733.7,310.859,481.301,1919.7,29.150,141.7,3.08,5.1,177.830,2.34,0.74
1959Q3,2775.488,1751.8,289.226,491.260,1916.4,29.350,140.5,3.82,5.3,178.657,2.74,1.09
1959Q4,2785.204,1753.7,299.356,484.052,1931.3,29.370,140.0,4.33,5.6,179.386,0.27,4.06
1960Q1,2847.699,1770.5,331.722,462.199,1955.5,29.540,139.6,3.50,5.2,180.007,2.31,1.19
...,...,...,...,...,...,...,...,...,...,...,...,...
2008Q3,13324.600,9267.7,1990.693,991.551,9838.3,216.889,1474.7,1.17,6.0,305.270,-3.16,4.33
2008Q4,13141.920,9195.3,1857.661,1007.273,9920.4,212.174,1576.5,0.12,6.9,305.952,-8.79,8.91
2009Q1,12925.410,9209.2,1558.494,996.287,9926.4,212.671,1592.8,0.22,8.1,306.547,0.94,-0.71


In [5]:
# Examine os alvos
experionml.plot_series()

In [6]:
experionml.plot_decomposition()

## Executar o pipeline

In [7]:
# Exogenous features are transformed normally
experionml.normalize()

Ajustando Normalizer...
Normalizando as variáveis...


In [8]:
experionml.warnings = True  # Let's turn on warnings for a sec

In [9]:
# Use o método apply para transformar as colunas alvo
experionml.apply(np.sqrt, columns=experionml.target)

Fitting FunctionTransformer...


/home/gerson/Projects/Personal/experionml/.venv/lib/python3.10/site-packages/pandas/core/internals/blocks.py:395: RuntimeWarning: invalid value encountered in sqrt
  result = func(self.values, **kwargs)


In [10]:
# Observe pelos avisos que talvez tenhamos NaNs no conjunto de dados agora
experionml.nans

realgdp      0
realcons     0
realinv      0
realgovt     0
realdpi      0
cpi          0
m1           0
tbilrate     0
unemp        0
pop          0
infl         6
realint     52
dtype: int64

In [11]:
# E, de fato, podemos vê-las nas colunas alvo
experionml.y

,infl,realint
Period,,
1959Q1,0.000000,0.000000
1959Q2,1.529706,0.860233
1959Q3,1.655295,1.044031
1959Q4,0.519615,2.014944
1960Q1,1.519868,1.090871
...,...,...
2008Q3,NaN,2.080865
2008Q4,NaN,2.984962
2009Q1,0.969536,NaN


In [12]:
# Impute os valores ausentes criados pela transformação
experionml.impute(strat_num="bfill", columns=experionml.target)

Ajustando Imputer...
Imputando valores ausentes...
 --> Imputando 6 valores ausentes com bfill na coluna infl.
 --> Imputando 52 valores ausentes com bfill na coluna realint.


In [13]:
experionml.y

,infl,realint
Period,,
1959Q1,0.000000,0.000000
1959Q2,1.529706,0.860233
1959Q3,1.655295,1.044031
1959Q4,0.519615,2.014944
1960Q1,1.519868,1.090871
...,...,...
2008Q3,0.969536,2.080865
2008Q4,0.969536,2.984962
2009Q1,0.969536,2.984962


In [14]:
experionml.run(["BATS", "MSTL"], n_trials=10, warnings=False)

ModuleNotFoundError: Unable to import the tbats package. Install it using `pip install tbats` or install all of experionml's optional dependencies with `pip install experionml[full]`.

## Analisar os resultados

In [ ]:
experionml.evaluate()

In [ ]:
with experionml.canvas():
    experionml.winner.plot_forecast(target=0)
    experionml.winner.plot_forecast(target=1)